# LeetCode 1215 — Stepping Numbers bằng Python

Link đề: https://leetcode.com/problems/stepping-numbers/ · Độ khó: **Medium** · Chủ đề: Math, Backtracking, Breadth-First Search

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề và chữ ký hàm được đối chiếu với bản mô tả công khai tại https://leetcode.doocs.org/en/lc/1215/ (method `countSteppingNumbers(low, high)` trả về **danh sách đã sắp xếp**, dù tên method có chữ "count").

Skeleton LeetCode:

```python
class Solution:
    def countSteppingNumbers(self, low: int, high: int) -> List[int]:
        ...
```

Ghi chú: kết quả là **list tăng dần** các stepping number trong đoạn `[low, high]` (tính cả hai đầu). Số `0` cũng là stepping number.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu định nghĩa **stepping number** (số bậc thang): mọi cặp chữ số kề nhau chênh lệch **đúng 1**; mọi số có một chữ số (kể cả `0`) đều thoả.
2. Nhận ra lối tư duy **"sinh thay vì lọc"** (generate instead of filter): đoạn `[low, high]` có thể dài `2·10^9`, nhưng số lượng stepping number nhỏ hơn rất nhiều → đi **xây** chúng từ từng chữ số.
3. Cài đặt được ba cách sinh: **BFS theo tầng** (tự nhiên ra thứ tự tăng), **DFS/backtracking** (phải sắp xếp lại), và **sinh sẵn + tìm kiếm nhị phân** (`bisect`).
4. Biết vì sao BFS từ các chữ số `1..9` cho ra dãy **tăng dần** và được phép **dừng sớm** (`break`) khi vượt `high`.
5. Tránh các lỗi: quên số `0`, bắt đầu từ `0` sinh ra số có số 0 đứng đầu, sinh chữ số `-1` hoặc `10`, quên sắp xếp kết quả DFS.

## 2. Đề bài & ví dụ

Một **stepping number** là số nguyên không âm mà mọi hai chữ số **kề nhau** có hiệu tuyệt đối **đúng bằng 1**. Ví dụ `321` là stepping number (`|3−2| = 1`, `|2−1| = 1`), còn `421` thì không (`|4−2| = 2`).

Cho hai số nguyên `low` và `high`, trả về **danh sách đã sắp xếp tăng dần** gồm tất cả stepping number trong đoạn `[low, high]`.

**Ví dụ 1:** `low = 0, high = 21` → `[0,1,2,3,4,5,6,7,8,9,10,12,21]`.

**Ví dụ 2:** `low = 10, high = 15` → `[10,12]`.

**Ràng buộc:** `0 <= low <= high <= 2 * 10^9`.

In [1]:
import bisect
import random
import time
from collections import deque
from functools import lru_cache
from typing import List

examples = [
    (0, 21, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 21]),
    (10, 15, [10, 12]),
]
for low, high, want in examples:
    print(f"low={low}, high={high} -> {want}")

low=0, high=21 -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 21]
low=10, high=15 -> [10, 12]


## 3. Kiến thức nền

### 3.1 Tách chữ số của một số nguyên

Hai cách phổ biến:
- Dùng chuỗi: `str(x)` rồi duyệt từng ký tự (dễ đọc).
- Dùng toán: `x % 10` là chữ số **cuối**, `x // 10` là bỏ chữ số cuối.

In [2]:
x = 4321
print("chuỗi:", [int(c) for c in str(x)])
digits = []
while x:
    digits.append(x % 10)
    x //= 10
print("toán (từ cuối lên):", digits)

chuỗi: [4, 3, 2, 1]
toán (từ cuối lên): [1, 2, 3, 4]


### 3.2 Kiểm tra một số có phải stepping number

Duyệt các cặp chữ số kề nhau, mọi cặp phải có hiệu tuyệt đối bằng 1. Hàm `zip(s, s[1:])` cho ra các cặp `(s[0], s[1]), (s[1], s[2]), ...`.

In [3]:
def is_stepping(x):
    s = str(x)
    return all(abs(int(a) - int(b)) == 1 for a, b in zip(s, s[1:]))


print([(v, is_stepping(v)) for v in [0, 7, 10, 11, 12, 321, 421, 98989]])

[(0, True), (7, True), (10, True), (11, False), (12, True), (321, True), (421, False), (98989, True)]


### 3.3 "Nối thêm một chữ số" vào cuối

Nếu `v` là số và `d` là một chữ số thì số mới là `v * 10 + d`. Ví dụ `12 * 10 + 3 = 123`. Muốn số mới vẫn là stepping number, chữ số `d` phải chênh **1** so với chữ số cuối `last = v % 10`, tức `d = last − 1` (nếu `last > 0`) hoặc `d = last + 1` (nếu `last < 9`).

In [4]:
def children(v):
    last = v % 10
    out = []
    if last > 0:
        out.append(v * 10 + last - 1)
    if last < 9:
        out.append(v * 10 + last + 1)
    return out


print(children(1), children(9), children(12), children(10))

[10, 12] [98] [121, 123] [101]


### 3.4 `deque` — hàng đợi hai đầu

`collections.deque` cho phép `append` (thêm cuối) và `popleft` (lấy đầu) đều `O(1)`. Đây là cấu trúc chuẩn cho **BFS** (Breadth-First Search — duyệt theo chiều rộng): phần tử vào trước được xử lý trước.

In [5]:
q = deque([1, 2, 3])
q.append(4)
print(q.popleft(), q.popleft(), list(q))

1 2 [3, 4]


### 3.5 `bisect` — tìm kiếm nhị phân trên list đã sắp xếp

- `bisect_left(a, x)`: vị trí đầu tiên có `a[i] >= x`.
- `bisect_right(a, x)`: vị trí đầu tiên có `a[i] > x`.

Vậy các phần tử trong đoạn `[low, high]` là `a[bisect_left(a, low) : bisect_right(a, high)]`.

In [6]:
a = [0, 1, 2, 10, 12, 21, 23, 32]
lo, hi = bisect.bisect_left(a, 5), bisect.bisect_right(a, 21)
print(lo, hi, a[lo:hi])

3 6 [10, 12, 21]


## 4. Ý tưởng nền tảng / trực giác

Liệt kê tay các stepping number theo **số chữ số**:

- 1 chữ số: `0, 1, 2, ..., 9` (10 số — không có cặp kề nào nên tự động thoả).
- 2 chữ số: chữ số đầu `1..9`, chữ số sau chênh 1: `10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 76, 78, 87, 89, 98` (17 số).
- 3 chữ số: nối thêm vào mỗi số 2 chữ số: `10 → 101`; `12 → 121, 123`; `21 → 210, 212`; ...

Quan sát: mỗi số có **tối đa 2 "con"** (thêm `last−1` hoặc `last+1`). Số có `k` chữ số nhiều nhất cỡ `9 · 2^(k−1)`. Với `high ≤ 2·10^9` (tối đa 10 chữ số) thì tổng số stepping number chỉ vài nghìn — **rất nhỏ** so với `2·10^9` số cần lọc.

Kiểm chứng bằng cách đếm tay (máy đếm hộ) cho các độ dài nhỏ:

In [7]:
for k in range(1, 5):
    lo = 0 if k == 1 else 10 ** (k - 1)
    cnt = sum(is_stepping(v) for v in range(lo, 10 ** k))
    print(f"{k} chữ số: {cnt} stepping number")

1 chữ số: 10 stepping number
2 chữ số: 17 stepping number
3 chữ số: 32 stepping number
4 chữ số: 61 stepping number


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** duyệt mọi số `v` từ `low` đến `high`, giữ lại số nào `is_stepping(v)`.

Kết quả tự nhiên đã tăng dần. Viết rất nhanh, dùng làm **trọng tài** cho các đoạn nhỏ.

In [8]:
def stepping_brute(low, high):
    return [v for v in range(low, high + 1) if is_stepping(v)]


for low, high, want in examples:
    got = stepping_brute(low, high)
    print(got)
    assert got == want

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 21]
[10, 12]


**Độ phức tạp:** Time `O((high − low + 1) · D)` với `D ≤ 10` là số chữ số. Khi `high − low = 2·10^9` thì cần cỡ `2·10^10` thao tác — **không thể** chạy kịp.

Thử đo: chỉ một đoạn dài `10^5` đã tốn đáng kể thời gian; nhân lên `20 000` lần là hết hy vọng.

In [9]:
t0 = time.perf_counter()
res = stepping_brute(0, 100_000)
dt = time.perf_counter() - t0
print(f"{len(res)} số trong [0, 1e5], mất {dt:.3f}s -> ước lượng cho 2e9: ~{dt * 20000 / 3600:.1f} giờ")

236 số trong [0, 1e5], mất 0.070s -> ước lượng cho 2e9: ~0.4 giờ


## 6. Bước đột phá

**Insight:** thay vì **lọc** `2·10^9` ứng viên, hãy **sinh** ra đúng các stepping number. Mỗi stepping number có ≥ 2 chữ số đều thu được từ một stepping number ngắn hơn bằng cách **nối thêm một chữ số** ở cuối (bỏ chữ số cuối của một stepping number vẫn được stepping number). Vậy toàn bộ tập hợp là một **cây** (rừng) có gốc là `1..9`:

- **Nếu** `last = v % 10 > 0` **thì** `v` có con `v*10 + last − 1`.
- **Nếu** `last < 9` **thì** `v` có con `v*10 + last + 1`.
- **Nếu** `v > high` **thì** mọi con của `v` còn lớn hơn (`≥ 10·v`) → **cắt tỉa** (pruning), không sinh tiếp.

Số `0` là trường hợp riêng: nó là stepping number nhưng **không** được làm gốc (nối vào sau `0` cho `01` = `1`, trùng lặp / số 0 đứng đầu).

**Bất biến của BFS:** hàng đợi luôn chứa các số theo thứ tự **tăng dần**. Do đó khi lấy ra một số `> high` thì mọi số sau nó cũng `> high` → được phép `break`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS từ các chữ số 1..9

Hàng đợi khởi tạo `1, 2, ..., 9`. Lấy lần lượt từng số; nếu vượt `high` thì dừng hẳn; nếu `≥ low` thì đưa vào đáp án; rồi đẩy các con vào cuối hàng đợi. Thêm `0` vào đầu đáp án nếu `low == 0`.

In [10]:
def stepping_bfs(low, high):
    ans = [0] if low == 0 else []
    q = deque(range(1, 10))
    while q:
        v = q.popleft()
        if v > high:
            break                     # mọi số còn lại trong hàng đợi đều lớn hơn
        if v >= low:
            ans.append(v)
        last = v % 10
        if last > 0:
            q.append(v * 10 + last - 1)
        if last < 9:
            q.append(v * 10 + last + 1)
    return ans


for low, high, want in examples:
    assert stepping_bfs(low, high) == want
print(stepping_bfs(100, 250))

[101, 121, 123, 210, 212, 232, 234]


**Vì sao hàng đợi tăng dần?** Các số được đẩy theo **tầng** (độ dài). Trong một tầng, cha tăng dần và mỗi cha đẩy con nhỏ (`last−1`) trước con lớn (`last+1`); con của cha nhỏ hơn luôn nhỏ hơn con của cha lớn hơn (vì `10·v + d` với `d ≤ 9` không thể vượt `10·(v+1)`). Tầng sau gồm số nhiều chữ số hơn nên lớn hơn mọi số tầng trước.

**Độ phức tạp:** gọi `S` là số stepping number `≤ high` (tối đa vài nghìn). Time `O(S)`, Space `O(S)` (hàng đợi + đáp án).

### 7.2 Cách 3 — DFS / backtracking rồi sắp xếp

Duyệt theo chiều sâu từ từng gốc `1..9`: đi sâu nối thêm chữ số đến khi vượt `high`. Thứ tự sinh ra **không** tăng dần (ví dụ `1, 10, 101, ..., 12, ...`), nên phải `sort` ở cuối. Viết bằng stack để khỏi lo giới hạn đệ quy (dù độ sâu tối đa chỉ 10).

In [11]:
def stepping_dfs(low, high):
    ans = [0] if low == 0 else []
    stack = list(range(9, 0, -1))
    while stack:
        v = stack.pop()
        if v > high:
            continue                  # DFS: KHÔNG được break, các nhánh khác vẫn có số nhỏ
        if v >= low:
            ans.append(v)
        last = v % 10
        if last < 9:
            stack.append(v * 10 + last + 1)
        if last > 0:
            stack.append(v * 10 + last - 1)
    ans.sort()
    return ans


for low, high, want in examples:
    assert stepping_dfs(low, high) == want
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


Bản đệ quy (backtracking) ngắn gọn và rất phổ biến — độ sâu đệ quy tối đa bằng số chữ số (≤ 10), hoàn toàn an toàn:

In [12]:
def stepping_dfs_rec(low, high):
    ans = [0] if low == 0 else []

    def dfs(v):
        if v > high:
            return
        if v >= low:
            ans.append(v)
        last = v % 10
        if last > 0:
            dfs(v * 10 + last - 1)
        if last < 9:
            dfs(v * 10 + last + 1)

    for d in range(1, 10):
        dfs(d)
    return sorted(ans)


for low, high, want in examples:
    assert stepping_dfs_rec(low, high) == want
print("Bản đệ quy cũng đúng")

Bản đệ quy cũng đúng


**Độ phức tạp:** Time `O(S log S)` vì phải sắp xếp, Space `O(S)`.

### 7.3 Cách 4 — Sinh sẵn toàn bộ một lần + `bisect`

Vì `high ≤ 2·10^9` cố định và tập stepping number nhỏ, ta có thể sinh **tất cả** stepping number `≤ 2·10^9` **một lần** (ví dụ ở cấp module/lớp), lưu thành list tăng dần. Mỗi truy vấn chỉ cần hai lần tìm kiếm nhị phân và cắt list. Rất hợp khi có **nhiều truy vấn**.

In [13]:
LIMIT = 2 * 10 ** 9
ALL_STEPPING = stepping_bfs(0, LIMIT)
print("Tổng số stepping number <= 2e9:", len(ALL_STEPPING))
print("vài số cuối:", ALL_STEPPING[-5:])


def stepping_bisect(low, high):
    i = bisect.bisect_left(ALL_STEPPING, low)
    j = bisect.bisect_right(ALL_STEPPING, high)
    return ALL_STEPPING[i:j]


for low, high, want in examples:
    assert stepping_bisect(low, high) == want
print("Cách 4 đúng trên ví dụ")

Tổng số stepping number <= 2e9: 3502
vài số cuối: [1234567676, 1234567678, 1234567876, 1234567878, 1234567898]
Cách 4 đúng trên ví dụ


**Độ phức tạp:** tiền xử lý `O(T)` với `T` là tổng số stepping number `≤ 2·10^9` (in ở trên — chỉ vài nghìn); mỗi truy vấn `O(log T + K)` với `K` là độ dài kết quả. Space `O(T)`.

### 7.4 Cách 5 (mở rộng) — Chỉ cần **đếm**: Digit DP

Nếu đề chỉ hỏi **số lượng** (như bài khó LeetCode 2801 với `high` có tới 100 chữ số), ta không thể liệt kê. Kỹ thuật **digit DP** (quy hoạch động theo chữ số) đếm số stepping number `≤ N` theo trạng thái `(vị trí, chữ số trước, còn bị chặn bởi N hay không, đã bắt đầu số chưa)`. Ở đây chỉ minh hoạ để so sánh với độ dài list:

In [14]:
def count_stepping_le(N):
    """Số stepping number trong [0, N] (N >= 0)."""
    s = str(N)

    @lru_cache(maxsize=None)
    def go(pos, prev, tight, started):
        if pos == len(s):
            return 1                      # mỗi cách điền hợp lệ = 1 số (kể cả số 0)
        limit = int(s[pos]) if tight else 9
        total = 0
        for d in range(limit + 1):
            nt = tight and d == limit
            if not started:
                total += go(pos + 1, d, nt, d != 0)   # vẫn đang là số 0 đứng đầu
            elif abs(d - prev) == 1:
                total += go(pos + 1, d, nt, True)
        return total

    return go(0, -1, True, False)


def count_stepping(low, high):
    return count_stepping_le(high) - (count_stepping_le(low - 1) if low > 0 else 0)


for low, high in [(0, 21), (10, 15), (0, 2 * 10 ** 9), (12345, 9876543)]:
    print((low, high), count_stepping(low, high), len(stepping_bfs(low, high)))
    assert count_stepping(low, high) == len(stepping_bfs(low, high))

(0, 21) 13 13
(10, 15) 2 2
(0, 2000000000) 3502 3502
(12345, 9876543) 734 734


**Độ phức tạp:** `O(D · 10 · 2 · 2 · 10)` với `D` là số chữ số — gần như hằng số. Đây là công cụ khi chỉ cần đếm với giới hạn khổng lồ.

## 8. TRACE từng bước

Trace BFS với `low = 10, high = 15`:

| Lấy ra v | v > 15? | v >= 10? | ans | Đẩy thêm | Hàng đợi sau bước |
|---|---|---|---|---|---|
| 1 | không | không | [] | 10, 12 | 2..9, 10, 12 |
| 2 | không | không | [] | 21, 23 | 3..9, 10, 12, 21, 23 |
| ... | | | | | |
| 9 | không | không | [] | 98 | 10, 12, 21, ..., 89, 98 |
| 10 | không | **có** | [10] | 101 | 12, 21, ..., 98, 101 |
| 12 | không | **có** | [10, 12] | 121, 123 | 21, ... |
| 21 | **có** → `break` | | | | |

Hàm `trace_bfs` in đúng các trạng thái này:

In [15]:
def trace_bfs(low, high, max_lines=20):
    ans = [0] if low == 0 else []
    q = deque(range(1, 10))
    step = 0
    while q:
        v = q.popleft()
        step += 1
        if v > high:
            print(f"bước {step:>2}: lấy {v:>4} > high={high} -> break")
            break
        if v >= low:
            ans.append(v)
        last = v % 10
        pushed = []
        if last > 0:
            pushed.append(v * 10 + last - 1)
        if last < 9:
            pushed.append(v * 10 + last + 1)
        q.extend(pushed)
        if step <= max_lines:
            print(f"bước {step:>2}: lấy {v:>4} | ans={ans} | đẩy {pushed} | đầu hàng đợi {list(q)[:4]}...")
    return ans


print(trace_bfs(10, 15))

bước  1: lấy    1 | ans=[] | đẩy [10, 12] | đầu hàng đợi [2, 3, 4, 5]...
bước  2: lấy    2 | ans=[] | đẩy [21, 23] | đầu hàng đợi [3, 4, 5, 6]...
bước  3: lấy    3 | ans=[] | đẩy [32, 34] | đầu hàng đợi [4, 5, 6, 7]...
bước  4: lấy    4 | ans=[] | đẩy [43, 45] | đầu hàng đợi [5, 6, 7, 8]...
bước  5: lấy    5 | ans=[] | đẩy [54, 56] | đầu hàng đợi [6, 7, 8, 9]...
bước  6: lấy    6 | ans=[] | đẩy [65, 67] | đầu hàng đợi [7, 8, 9, 10]...
bước  7: lấy    7 | ans=[] | đẩy [76, 78] | đầu hàng đợi [8, 9, 10, 12]...
bước  8: lấy    8 | ans=[] | đẩy [87, 89] | đầu hàng đợi [9, 10, 12, 21]...
bước  9: lấy    9 | ans=[] | đẩy [98] | đầu hàng đợi [10, 12, 21, 23]...
bước 10: lấy   10 | ans=[10] | đẩy [101] | đầu hàng đợi [12, 21, 23, 32]...
bước 11: lấy   12 | ans=[10, 12] | đẩy [121, 123] | đầu hàng đợi [21, 23, 32, 34]...
bước 12: lấy   21 > high=15 -> break
[10, 12]


Trace thêm thứ tự sinh của DFS để thấy vì sao **phải sắp xếp**:

In [16]:
def trace_dfs_order(high):
    order = []
    stack = list(range(9, 0, -1))
    while stack:
        v = stack.pop()
        if v > high:
            continue
        order.append(v)
        last = v % 10
        if last < 9:
            stack.append(v * 10 + last + 1)
        if last > 0:
            stack.append(v * 10 + last - 1)
    return order


o = trace_dfs_order(130)
print("thứ tự DFS sinh ra:", o)
print("đã tăng dần chưa?", o == sorted(o))

thứ tự DFS sinh ra: [1, 10, 101, 12, 121, 123, 2, 21, 23, 3, 32, 34, 4, 43, 45, 5, 54, 56, 6, 65, 67, 7, 76, 78, 8, 87, 89, 9, 98]
đã tăng dần chưa? False


## 9. Code nộp LeetCode

Dùng BFS (Cách 2): ngắn, kết quả tự động tăng dần, có cắt tỉa sớm.

In [17]:
class Solution:
    def countSteppingNumbers(self, low: int, high: int) -> List[int]:
        ans = [0] if low == 0 else []
        q = deque(range(1, 10))
        while q:
            v = q.popleft()
            if v > high:
                break
            if v >= low:
                ans.append(v)
            last = v % 10
            if last > 0:
                q.append(v * 10 + last - 1)
            if last < 9:
                q.append(v * 10 + last + 1)
        return ans


sol = Solution()
for low, high, want in examples:
    got = sol.countSteppingNumbers(low, high)
    print(got)
    assert got == want

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 21]
[10, 12]


## 10. Vì sao đúng

1. **Đầy đủ:** mọi stepping number `x ≥ 10` có dạng `x = 10·p + d` với `p = x // 10` cũng là stepping number (bỏ chữ số cuối không phá điều kiện các cặp còn lại) và `|d − p % 10| = 1`. Quy nạp theo số chữ số: `p` được sinh ra (gốc `1..9` là cơ sở), nên `x` được sinh ra từ `p`. Số `0` được thêm riêng.
2. **Không thừa:** mỗi số được sinh bằng cách nối chữ số chênh 1 với chữ số cuối, nên luôn là stepping number; và mỗi số có **đúng một cha** (`x // 10`) nên không bị sinh trùng.
3. **Cắt tỉa an toàn:** nếu `v > high` thì mọi con `≥ 10·v > high`, mọi hậu duệ cũng vậy.
4. **`break` an toàn trong BFS:** hàng đợi tăng dần (mục 7.1), nên số đầu hàng đợi `> high` kéo theo mọi số còn lại `> high`.
5. **Có thứ tự:** BFS lấy ra theo thứ tự tăng dần, ta chỉ `append` khi `v ≥ low` → `ans` tăng dần. Riêng `0` nhỏ nhất nên đứng đầu.

## 11. Time / Space Complexity

Gọi `S` là số stepping number `≤ high`. Với `high ≤ 2·10^9`, `S` chỉ cỡ vài nghìn (in ở mục 7.3).

| Cách | Time | Space |
|---|---|---|
| 1. Lọc từng số | `O((high − low) · D)` — quá chậm | `O(kết quả)` |
| 2. BFS | `O(S)` | `O(S)` |
| 3. DFS + sort | `O(S log S)` | `O(S)` |
| 4. Sinh sẵn + bisect | tiền xử lý `O(T)`, truy vấn `O(log T + K)` | `O(T)` |
| 5. Digit DP (chỉ đếm) | `O(D · 10 · 4 · 10)` | `O(D · 40)` |

`D` = số chữ số của `high` (≤ 10), `T` = tổng stepping number `≤ 2·10^9`, `K` = độ dài kết quả.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
methods = {
    "brute": stepping_brute,
    "bfs": stepping_bfs,
    "dfs": stepping_dfs,
    "dfs_rec": stepping_dfs_rec,
    "bisect": stepping_bisect,
    "submit": lambda lo, hi: Solution().countSteppingNumbers(lo, hi),
}
edge = [
    (0, 0, [0]),
    (1, 1, [1]),
    (11, 11, []),                 # 11 không phải stepping (|1-1| = 0)
    (13, 20, []),
    (9, 10, [9, 10]),
    (98, 101, [98, 101]),
    (0, 9, list(range(10))),
]
for name, f in methods.items():
    for low, high, want in examples + edge:
        assert f(low, high) == want, (name, low, high)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [19]:
rng = random.Random(1215)
for _ in range(300):
    low = rng.randint(0, 50_000)
    high = low + rng.randint(0, 3_000)
    want = stepping_brute(low, high)
    for name, f in methods.items():
        if name != "brute":
            assert f(low, high) == want, (name, low, high)
    assert count_stepping(low, high) == len(want)
print("300 đoạn ngẫu nhiên: mọi cách khớp brute force")

300 đoạn ngẫu nhiên: mọi cách khớp brute force


### 12.3 Test giới hạn

In [20]:
for name in ["bfs", "dfs", "dfs_rec", "bisect", "submit"]:
    t0 = time.perf_counter()
    r = methods[name](0, 2 * 10 ** 9)
    dt = time.perf_counter() - t0
    assert r == ALL_STEPPING and r == sorted(r) and all(is_stepping(v) for v in r)
    print(f"{name:8s} {len(r)} số, {dt * 1000:.1f} ms")
assert Solution().countSteppingNumbers(2 * 10 ** 9, 2 * 10 ** 9) == []
assert Solution().countSteppingNumbers(1234567898, 1234567898) == [1234567898]

bfs      3502 số, 0.7 ms
dfs      3502 số, 1.0 ms
dfs_rec  3502 số, 1.0 ms
bisect   3502 số, 0.0 ms
submit   3502 số, 0.7 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên số 0

In [21]:
def wrong_no_zero(low, high):
    ans = []
    q = deque(range(1, 10))
    while q:
        v = q.popleft()
        if v > high:
            break
        if v >= low:
            ans.append(v)
        last = v % 10
        if last > 0:
            q.append(v * 10 + last - 1)
        if last < 9:
            q.append(v * 10 + last + 1)
    return ans


print("sai :", wrong_no_zero(0, 5))
print("đúng:", stepping_bfs(0, 5))

sai : [1, 2, 3, 4, 5]
đúng: [0, 1, 2, 3, 4, 5]


**Vì sao sai:** `0` là số một chữ số nên là stepping number (ví dụ 1 của đề có `0`). Gốc BFS là `1..9` nên `0` không bao giờ được sinh ra. **Sửa:** `ans = [0] if low == 0 else []`.

### Lỗi 2 — Cho cả 0 làm gốc BFS

In [22]:
def wrong_root_zero(low, high):
    ans = []
    q = deque(range(0, 10))
    while q:
        v = q.popleft()
        if v > high:
            break
        if v >= low:
            ans.append(v)
        last = v % 10
        if last > 0:
            q.append(v * 10 + last - 1)
        if last < 9:
            q.append(v * 10 + last + 1)
    return ans


print("sai :", wrong_root_zero(0, 12))
print("đúng:", stepping_bfs(0, 12))

sai : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 10, 12]
đúng: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]


**Vì sao sai:** con của `0` là `0·10 + 1 = 1` — tức "01", trùng với số `1` đã có; kết quả có phần tử lặp và không còn tăng dần chặt. **Sửa:** chỉ dùng `1..9` làm gốc, thêm `0` riêng.

### Lỗi 3 — Không kiểm tra biên khi nối chữ số

In [23]:
def wrong_children(v):
    last = v % 10
    return [v * 10 + last - 1, v * 10 + last + 1]     # quên điều kiện 0 < last < 9


print("con sai của 9 :", wrong_children(9), "-> 9*10+10 =", 9 * 10 + 10, "không phải 'nối chữ số 10'")
print("con sai của 10:", wrong_children(10), "-> 10*10-1 =", 10 * 10 - 1)
print("is_stepping(100)?", is_stepping(100), "| is_stepping(99)?", is_stepping(99))
print("con đúng:", children(9), children(10))

con sai của 9 : [98, 100] -> 9*10+10 = 100 không phải 'nối chữ số 10'
con sai của 10: [99, 101] -> 10*10-1 = 99
is_stepping(100)? False | is_stepping(99)? False
con đúng: [98] [101]


**Vì sao sai:** chữ số chỉ từ `0` đến `9`. Với `last = 9`, "chữ số" `10` làm phép tính tràn sang hàng chục (`9·10 + 10 = 100` — không phải stepping); với `last = 0`, `−1` cho `99`. **Sửa:** chỉ thêm `last − 1` khi `last > 0`, chỉ thêm `last + 1` khi `last < 9`.

### Lỗi 4 — DFS nhưng `break` hoặc quên sắp xếp

In [24]:
def wrong_dfs_unsorted(low, high):
    ans = [0] if low == 0 else []
    stack = list(range(9, 0, -1))
    while stack:
        v = stack.pop()
        if v > high:
            break                     # sai: DFS không có tính tăng dần
        if v >= low:
            ans.append(v)
        last = v % 10
        if last < 9:
            stack.append(v * 10 + last + 1)
        if last > 0:
            stack.append(v * 10 + last - 1)
    return ans                        # sai: quên sort


print("sai :", wrong_dfs_unsorted(0, 21))
print("đúng:", stepping_dfs(0, 21))

sai : [0, 1, 10]
đúng: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 21]


**Vì sao sai:** thứ tự DFS là `1, 10, 101, ...` — gặp `101 > 21` thì `break` khiến bỏ sót cả `2..9, 12, 21`. Dù không `break`, thứ tự vẫn không tăng dần. **Sửa:** trong DFS dùng `continue` (hoặc `return` trong đệ quy) và `sort` cuối cùng; hoặc dùng BFS.

## 14. Mẹo & nhận dạng pattern

- Miền giá trị khổng lồ (`10^9`) nhưng tập "số đặc biệt" thưa → **sinh** (BFS/DFS từng chữ số) thay vì lọc.
- Ràng buộc chỉ liên quan **chữ số kề nhau** → trạng thái chỉ cần "chữ số cuối" → dễ sinh bằng cây hoặc đếm bằng digit DP.
- BFS từ các gốc tăng dần, con sinh theo thứ tự tăng → hàng đợi **tự động có thứ tự**, được `break` sớm.
- Nhiều truy vấn trên cùng tập cố định → **tiền xử lý + bisect**.
- Chỉ hỏi **bao nhiêu** và giới hạn tính bằng số chữ số → **digit DP**.

## 15. Biến thể

1. **Chênh lệch đúng `k`** thay vì 1 (ví dụ LeetCode 967 "Numbers With Same Consecutive Differences": mọi số có đúng `n` chữ số, chữ số kề chênh `k`). Cẩn thận khi `k = 0`: hai con trùng nhau.
2. **Đếm** stepping number trong `[low, high]` với `low, high` là chuỗi tới 100 chữ số, lấy modulo `10^9 + 7` (LeetCode 2801, Hard) → digit DP.
3. Số có các chữ số **tăng dần liên tiếp** (`123`, `4567` — LeetCode 1291 Sequential Digits): mỗi số chỉ có đúng một con.
4. Stepping number **vòng**: thêm điều kiện chữ số đầu và cuối cũng chênh 1.

In [25]:
def same_consecutive_diff(n, k):
    """LeetCode 967: mọi số có đúng n chữ số (không có số 0 đứng đầu, trừ khi n = 1), chữ số kề chênh k."""
    level = list(range(1, 10)) if n > 1 else list(range(10))
    for _ in range(n - 1):
        nxt = []
        for v in level:
            last = v % 10
            for d in {last - k, last + k}:      # set để k = 0 không bị trùng
                if 0 <= d <= 9:
                    nxt.append(v * 10 + d)
        level = nxt
    return sorted(level)


print(same_consecutive_diff(3, 7))
print(same_consecutive_diff(2, 0))

[181, 292, 707, 818, 929]
[11, 22, 33, 44, 55, 66, 77, 88, 99]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lọc | kiểm tra từng số | `O((high−low)·D)` | `O(K)` | trọng tài, đoạn nhỏ |
| 2. BFS | sinh theo tầng từ 1..9 | `O(S)` | `O(S)` | **nộp bài**: có thứ tự sẵn, dừng sớm |
| 3. DFS + sort | sinh theo chiều sâu | `O(S log S)` | `O(S)` | quen backtracking, code đệ quy ngắn |
| 4. Sinh sẵn + bisect | tiền xử lý một lần | `O(log T + K)`/truy vấn | `O(T)` | nhiều truy vấn |
| 5. Digit DP | đếm theo chữ số | `O(D·400)` | `O(D·40)` | chỉ cần đếm, giới hạn cực lớn |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`Solution().countSteppingNumbers(20, 35)` trả về gì?

### Bài 2 (tính tay)

Liệt kê tất cả stepping number có đúng 3 chữ số bắt đầu bằng chữ số `9`. Có bao nhiêu số?

### Bài 3 (điền TODO) — đếm số stepping number có đúng `k` chữ số

Dùng DP: `dp[d]` = số stepping number có độ dài hiện tại kết thúc bằng chữ số `d`.

In [26]:
def count_k_digits(k):
    if k == 1:
        return 10
    dp = [0] + [1] * 9          # độ dài 1, không tính số 0 làm chữ số đầu
    for _ in range(k - 1):
        # TODO: new[d] = dp[d-1] + dp[d+1] (chú ý biên d = 0 và d = 9)
        raise NotImplementedError
    return sum(dp)

# print([count_k_digits(k) for k in range(1, 5)])   # mong đợi [10, 17, 32, 61]

### Bài 4 (tự viết lại)

Viết lại lời giải BFS nhưng **theo tầng** (mỗi vòng xử lý cả một danh sách `level`, sinh `next_level`), không dùng `deque`.

In [27]:
def stepping_by_level_todo(low, high):
    # TODO: ans = [0] nếu low == 0; level = [1..9]; lặp đến khi level rỗng
    raise NotImplementedError

# print(stepping_by_level_todo(0, 21))   # mong đợi [0, 1, ..., 9, 10, 12, 21]

### Bài 5 (mini challenge) — stepping number lớn nhất `≤ N`

Viết `largest_stepping_le(N)` trả về stepping number lớn nhất không vượt quá `N` (`0 ≤ N ≤ 2·10^9`). Yêu cầu không duyệt lùi từng số.

In [28]:
def largest_stepping_le_todo(N):
    # TODO: gợi ý — dùng ALL_STEPPING và bisect
    raise NotImplementedError

# print(largest_stepping_le_todo(100))   # mong đợi 98

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: chỉ có số 2 chữ số. Mức 2: chữ số đầu 2 hoặc 3. Mức 3: `21, 23, 32, 34` — số nào `≤ 35`?

**Bài 2.** Mức 1: chữ số thứ hai chỉ có thể là `8`. Mức 2: chữ số thứ ba là `7` hoặc `9`. Mức 3: `987, 989` → 2 số.

**Bài 3.** Mức 1: tạo `new = [0] * 10`. Mức 2: `new[d] = (dp[d-1] if d > 0 else 0) + (dp[d+1] if d < 9 else 0)`. Mức 3: gán `dp = new` sau mỗi vòng.

**Bài 4.** Mức 1: `for v in level`. Mức 2: nếu `v > high` thì bỏ qua (có thể dừng sớm vì `level` tăng dần). Mức 3: dừng khi `level` rỗng.

**Bài 5.** Mức 1: `bisect_right(ALL_STEPPING, N)` cho vị trí đầu tiên `> N`. Mức 2: phần tử ngay trước nó là đáp án. Mức 3: luôn tồn tại vì `0 ≤ N` và `0` là stepping.

## 19. Đáp án tham khảo

In [29]:
# Bài 1, 2
print(Solution().countSteppingNumbers(20, 35))
assert Solution().countSteppingNumbers(20, 35) == [21, 23, 32, 34]
three_digit_9 = [v for v in Solution().countSteppingNumbers(900, 999)]
print(three_digit_9)
assert three_digit_9 == [987, 989]

[21, 23, 32, 34]
[987, 989]


In [30]:
def count_k_digits(k):
    if k == 1:
        return 10
    dp = [0] + [1] * 9
    for _ in range(k - 1):
        new = [0] * 10
        for d in range(10):
            new[d] = (dp[d - 1] if d > 0 else 0) + (dp[d + 1] if d < 9 else 0)
        dp = new
    return sum(dp)


def stepping_by_level(low, high):
    ans = [0] if low == 0 else []
    level = list(range(1, 10))
    while level:
        nxt = []
        for v in level:
            if v > high:
                break                 # level tăng dần
            if v >= low:
                ans.append(v)
            last = v % 10
            if last > 0:
                nxt.append(v * 10 + last - 1)
            if last < 9:
                nxt.append(v * 10 + last + 1)
        level = nxt
    return ans


def largest_stepping_le(N):
    return ALL_STEPPING[bisect.bisect_right(ALL_STEPPING, N) - 1]


assert [count_k_digits(k) for k in range(1, 5)] == [10, 17, 32, 61]
for k in range(1, 6):
    lo = 0 if k == 1 else 10 ** (k - 1)
    assert count_k_digits(k) == len(stepping_bfs(lo, 10 ** k - 1))
rng = random.Random(5)
for _ in range(200):
    lo = rng.randint(0, 10 ** 6)
    hi = lo + rng.randint(0, 10 ** 5)
    assert stepping_by_level(lo, hi) == stepping_bfs(lo, hi)
assert largest_stepping_le(100) == 98 and largest_stepping_le(0) == 0 and largest_stepping_le(21) == 21
for N in [rng.randint(0, 10 ** 5) for _ in range(100)]:
    assert largest_stepping_le(N) == max(v for v in ALL_STEPPING if v <= N)
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
ans = [0] if low == 0 else []          # số 0 xử lý riêng
q = deque(range(1, 10))                # gốc 1..9, KHÔNG có 0
while q:
    v = q.popleft()
    if v > high: break                 # BFS tăng dần -> dừng sớm
    if v >= low: ans.append(v)
    last = v % 10
    if last > 0: q.append(v * 10 + last - 1)
    if last < 9: q.append(v * 10 + last + 1)
return ans
```

**Checklist:**
- [ ] Có số `0` khi `low == 0`?
- [ ] Gốc chỉ là `1..9` (không sinh số có 0 đứng đầu, không trùng)?
- [ ] Kiểm tra biên `last > 0`, `last < 9` trước khi nối?
- [ ] Kết quả **tăng dần** (BFS tự có; DFS phải `sort`)?
- [ ] DFS dùng `continue`/`return`, không `break` khi vượt `high`?
- [ ] Thử: `low = high = 0`, đoạn rỗng (`13..20`), `high = 2·10^9`?